# 📶 24-信号处理 · 00 信号与系统基础

> 关键词：信号、LTI 系统、卷积、脉冲响应
> 面向**算法 / 语音 / 音频方向秋招**的第一课：把「信号」和「系统」用数学语言说清楚，**卷积怎么算**是全文主线。

## 本节要解决的问题

| # | 问题 | 对应内容 |
| --- | --- | --- |
| 1 | 什么是信号？连续/离散怎么分？ | §1 核心概念 |
| 2 | LTI 系统为什么是信号处理的"基本盘"？ | §2 深入原理 |
| 3 | 卷积 y = x*h 到底怎么逐点算？ | §3 手算 ✍️ |
| 4 | 怎么用 numpy 验证手算？ | §4 代码实战 |

## 故事引入

你对着手机说了一句"你好"，麦克风把声波变成电压、ADC 以 16kHz 采成一串数字。这串数字就是**离散信号**；手机里的降噪、回声消除、ASR 前端，全是围绕它做**系统**运算。整章就是"数字从哪来、怎么被加工"。

开始讨论信号的分类之前，先看一段**真实的信号**。下图是人体心电活动（ECG）的原始记录，每个 QRS 波群对应一次心跳。它提醒我们：真实信号既不是纯净的正弦、也不是理想的脉冲——有基线漂移、有肌电噪声、有心率变化，而这些"不干净"恰恰是后续所有处理（滤波、频谱、特征提取）的起点。

![开始讨论信号的分类之前，先看一段**真实的信号**。下图是人](../figures/ecg_waveform.jpg)

> **怎么读**：横轴是时间，纵轴是心电电位。可以看到周期性的 P-QRS-T 复合波——这就是"准周期非平稳信号"的典型样本，与本课讲的连续/离散、确定性/随机信号分类一一对应。
>
> 📚 **图源**：Jia Y et al. (2024) *Preprocessing and Denoising Techniques for Electrocardiography and Magnetocardiography: A Review*, Bioengineering 11(11):1109（开放获取，CC-BY 4.0）图 1。


## 核心概念

### (a) 信号的两大维度：连续/离散 × 时间/取值

| 连续时间 | 离散时间 |
| --- | --- | --- |
| **连续取值** | 模拟信号 x(t) | 采样后的数字信号 x[n] |
| **离散取值** | 量化(ADC) | 数字信号处理 DSP |

**连续**指自变量（时间）连续；**离散**指 n 只在整数上有定义。本章所有手算都在离散域。

### (b) 系统的两大性质：线性 + 时不变

- **线性**：$$T\{ax_1+bx_2\}=aT\{x_1\}+bT\{x_2\}$$
- **时不变**：输入平移 k，输出同样平移 k：$y[n-k]=T\{x[n-k]\}$

**同时满足 = LTI（线性时不变）系统**——信号处理 90% 的公式都建立在 LTI 上。

## 深入原理：LTI 系统与卷积

**脉冲响应** $h[n]$：输入单位冲激 $\delta[n]$ 时系统的输出。

LTI 系统的核心定理：**输出 = 输入与脉冲响应的卷积**

$$y[n]=\sum_{k=-\infty}^{\infty}x[k]\,h[n-k]=(x*h)[n]$$

直觉：把输入 $x$ 拆成无数个单位冲激 $\delta[n-k]$（每个幅度 $x[k]$），每个冲激经过系统变成 $x[k]h[n-k]$，全部叠加就是输出。卷积的三个等价做法：翻褶（fold）、移位（shift）、相乘相加（multiply-add）。

## ✍️ 手算卷积：x=[1,2,3]，h=[1,1]

**设定**：输入 $x=[1,2,3]$，系统脉冲响应 $h=[1,1]$（两拍平均器）。

**逐点展开**（$y[n]=\sum_k x[k]h[n-k]$）：

$$\begin{aligned} y[0]&=x[0]h[0]=1\times1=\mathbf{1}\\ y[1]&=x[1]h[0]+x[0]h[1]=2+1=\mathbf{3}\\ y[2]&=x[2]h[0]+x[1]h[1]=3+2=\mathbf{5}\\ y[3]&=x[2]h[1]=3\end{aligned}$$

**结果**：$y=[1,3,5,3]$，长度 $L_x+L_h-1=3+2-1=4$。

**手算收获**：

- 输出长度 = 输入长度 + 滤波器长度 − 1——**卷积会"拉长"信号**，滤波后长度变化是常见考点
- 每个 $y[n]$ 是"输入与翻褶后的 $h$ 对齐相乘再求和"：$y[1]$ 用了 $x[1]h[0]$ 和 $x[0]h[1]$ 两项，越往中间项越多
- 交换律 $x*h=h*x$、结合律、分配律是 LTI 三大性质：串联系统 $h_1*h_2$、并联 $h_1+h_2$（§面试考点）
- 如果 $h=[1,1]$ 是"两拍平均"，那 $y$ 就是平滑后的 $x$——**卷积即滤波**，这就是 §04 滤波器的伏笔

In [1]:
# ---- §3.1 验证：numpy 复算卷积 ----
import numpy as np
x = np.array([1.0, 2.0, 3.0])
h = np.array([1.0, 1.0])
y = np.convolve(x, h)
print('y = x * h =', y.tolist(), '，长度 =', len(y))
assert y.tolist() == [1.0, 3.0, 5.0, 3.0] and len(y) == 4
print('\n✓ 与手算一致')


y = x * h = [1.0, 3.0, 5.0, 3.0] ，长度 = 4

✓ 与手算一致


## 代码实战：自己写一遍卷积

下面不用 `np.convolve`，手写双循环复现定义，再与 numpy 对照——数字与你上面手算的 $[1,3,5,3]$ 完全一致。

In [2]:
# ---- 手写卷积双循环 ----
def my_convolve(x, h):
    L = len(x) + len(h) - 1
    y = np.zeros(L)
    for n in range(L):
        for k in range(len(x)):
            if 0 <= n - k < len(h):
                y[n] += x[k] * h[n - k]
    return y

x = np.array([1.0, 2.0, 3.0])
h = np.array([1.0, 1.0])
print('手写卷积:', my_convolve(x, h).tolist())
print('np.convolve:', np.convolve(x, h).tolist())
assert np.allclose(my_convolve(x, h), np.convolve(x, h))
print('✓ 与 numpy 一致')


手写卷积: [1.0, 3.0, 5.0, 3.0]
np.convolve: [1.0, 3.0, 5.0, 3.0]
✓ 与 numpy 一致


## 面试考点

**Q1：卷积和相关的区别？**

相关（correlation）不翻褶 $h$：$r[n]=\sum_k x[k]h[n+k]$。卷积用于系统响应，相关用于相似度匹配（模板匹配/互相关对齐）。

**Q2：两个 LTI 系统串联/并联的等效脉冲响应？**

串联 = 卷积 $h_1*h_2$（顺序可交换）；并联 = 相加 $h_1+h_2$。

**Q3：卷积的时间复杂度？**

直接 $O(L_xL_h)$；用 FFT 加速到 $O(N\log N)$（$N\ge L_x+L_h-1$）——这是 §01 的动机。

## 小结与下一章预告

- 信号分连续/离散；**LTI = 线性 + 时不变**
- LTI 输出 = 输入与脉冲响应**卷积**；$y=[1,3,5,3]$ 手算见证全过程
- 下一章：把信号从时域变到频域——**DFT/FFT**，卷积在频域将变成乘法

## ✅ 自测清单

- [ ] 能说出线性、时不变各自的含义并举例
- [ ] 能手算两个短序列的卷积（长度 Lx+Lh-1）
- [ ] 知道串联=卷积、并联=相加
- [ ] 能说出直接卷积与 FFT 卷积的复杂度对比